## Training Data

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder # Used for converting categorical variables into binary format
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

### Loading training data. We split our training data into two parts. First 400 are used as training data, next 150 are used as validation data.

In [4]:
train_data = pd.read_csv('train_data.csv',index_col=0)

train_data.loc[train_data['OCCUPATION']!=1, 'OCCUPATION'] = 0 # Convert non-occupation to 0
tr_data = train_data[0:400] # Select the first 400 rows for training
va_data = train_data[400:] # Select rows from 400 onwards for validation
va_data.index = range(150)

In [5]:
train_data

,DEBT,YRS_IN_RESIDENT,AGE,YRS_OF_EMPLOYMENT,DTI,NUM_PREV_APP,OCCUPATION,PROVIDED_SIN,MARRIAGE,INCOME,EDUCATION,CREDIT_PROFILE,APPROVAL_STATUS
0,1600,3,19,1.50,2.000,0,0,1,1,13,4,1,0
1,2200,1,48,4.25,0.125,0,1,1,2,7,4,1,1
2,600,2,35,4.50,5.750,0,1,1,2,14,7,0,1
3,200,4,20,0.50,0.000,3,0,1,2,1,1,0,0
4,100,8,45,7.00,1.625,0,0,1,1,8,4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
545,0,9,17,0.00,0.040,0,0,1,1,8,4,0,0
546,1200,1,29,3.50,3.500,3,1,1,2,9,4,1,0
547,100,5,27,14.50,3.085,1,0,0,2,14,7,1,1
548,300,8,32,16.25,3.000,9,0,1,2,2,4,1,1


### Prepping Data

In [7]:
# first, we list all the categorical variables to be one hot encoded
cat_vars = ['MARRIAGE', 'EDUCATION']

In [8]:
encoders = [OneHotEncoder(categories='auto') for _ in range(len(cat_vars))] # Create OneHotEncoders for each categorical variable

# fitting and transforming training data for each variable and transforming again using fitted encorders
encoded_tr = [encoders[i].fit_transform(tr_data[[cat_var]]).todense() for i,cat_var in enumerate(cat_vars)]
encoded_va = [encoders[i].transform(va_data[[cat_var]]).todense() for i,cat_var in enumerate(cat_vars)]

In [9]:
# aggregating our data with the one hot encoded data
# drop the Label column and also drop the cat_vars
# this way we can join the encoded categorical variables with the continuous variables
X_train = pd.concat([tr_data.iloc[:,:-1].drop(cat_vars, axis=1),
                     pd.DataFrame(np.concatenate(encoded_tr, axis=1))], axis=1)
X_validation = pd.concat([va_data.iloc[:,:-1].drop(cat_vars, axis=1),
                          pd.DataFrame(np.concatenate(encoded_va, axis=1))], axis=1)
y_train = tr_data.iloc[:,-1]
y_validation = va_data.iloc[:,-1]
X_train = X_train.rename(columns={0: 'Marriage 1',1: 'Marriage 2' ,2: 'Marriage 3' ,3: 'Edu 1',4: 'Edu 2' ,5: 'Edu 3',
                                  6: 'Edu 4' ,7: 'Edu 5',8: 'Edu 6' ,9: 'Edu 7'})
X_validation = X_validation.rename(columns={0: 'Marriage 1',1: 'Marriage 2' ,2: 'Marriage 3',3: 'Edu 1',4: 'Edu 2',5: 'Edu 3',
                                            6: 'Edu 4',7: 'Edu 5',8: 'Edu 6' ,9: 'Edu 7'})

In [10]:
X_train.head()

,DEBT,YRS_IN_RESIDENT,AGE,YRS_OF_EMPLOYMENT,DTI,NUM_PREV_APP,OCCUPATION,PROVIDED_SIN,INCOME,CREDIT_PROFILE,Marriage 1,Marriage 2,Marriage 3,Edu 1,Edu 2,Edu 3,Edu 4,Edu 5,Edu 6,Edu 7
0,1600,3,19,1.50,2.000,0,0,1,13,1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2200,1,48,4.25,0.125,0,1,1,7,1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,600,2,35,4.50,5.750,0,1,1,14,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,200,4,20,0.50,0.000,3,0,1,1,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,100,8,45,7.00,1.625,0,0,1,8,0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [11]:
# Standardize specified columns (0, 1, 2, 3, 4, 5, 8) by removing the mean and scaling to unit variance
for i in [0,1,2,3,4,5,8]:
    X1 = X_train.iloc[:,i]
    mean = X1.mean()
    std = X1.std()
    X_train.iloc[:,i] = (X1-mean)/std
    X_validation.iloc[:,i] = (X_validation.iloc[:,i]-mean)/std

/var/folders/1h/mhjxr6bx0yg08w73dbfxx3vh0000gn/T/ipykernel_48038/2573608646.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0      1.303140
1      1.993698
2      0.152210
3     -0.308161
4     -0.423254
         ...   
395   -0.423254
396   -0.538347
397    0.152210
398   -0.538347
399    0.037117
Name: DEBT, Length: 400, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_train.iloc[:,i] = (X1-mean)/std
/var/folders/1h/mhjxr6bx0yg08w73dbfxx3vh0000gn/T/ipykernel_48038/2573608646.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0      0.612582
1     -0.193068
2      0.612582
3      0.152210
4     -0.077976
         ...   
145   -0.538347
146    0.842768
147   -0.423254
148   -0.193068
149    0.152210
Name: DEBT, Length: 150, dtype: float64' has dtype incompatible with int64, please e

In [12]:
X_train.head()

,DEBT,YRS_IN_RESIDENT,AGE,YRS_OF_EMPLOYMENT,DTI,NUM_PREV_APP,OCCUPATION,PROVIDED_SIN,INCOME,CREDIT_PROFILE,Marriage 1,Marriage 2,Marriage 3,Edu 1,Edu 2,Edu 3,Edu 4,Edu 5,Edu 6,Edu 7
0,1.303140,-0.673039,-1.080653,-0.629013,-0.075700,-0.566394,0,1,1.540430,1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,1.993698,-1.358938,1.307986,-0.075482,-0.664258,-0.566394,1,1,-0.109294,1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.152210,-1.015989,0.237217,-0.025161,1.101415,-0.566394,1,1,1.815384,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,-0.308161,-0.330089,-0.998287,-0.830297,-0.703495,0.214839,0,1,-1.759018,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.423254,1.041710,1.060885,0.478050,-0.193412,-0.566394,0,1,0.165660,0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


### We use the regularized logistic regression to see results

In [14]:
lr = LogisticRegression(penalty='l1',solver='liblinear',C=1)
lr.fit(X_train, y_train)

LogisticRegression(C=1, penalty='l1', solver='liblinear')

In [15]:
print('Intercept = ',lr.intercept_[0])
# match coefs with categories
Ind=[i for i,x in enumerate(lr.coef_.flatten()) if x!=0]
dict(zip(X_train.columns[Ind],lr.coef_.flatten()[Ind]))

Intercept =  -2.09392390535329


{'DEBT': 0.11765992420585293,
 'YRS_IN_RESIDENT': 0.041790358514883386,
 'AGE': -0.0065911316786864075,
 'YRS_OF_EMPLOYMENT': -0.11589601285939095,
 'DTI': 0.16909690925466314,
 'NUM_PREV_APP': 0.6552032101440812,
 'PROVIDED_SIN': -0.18425891795879384,
 'INCOME': 0.573971542022012,
 'CREDIT_PROFILE': 3.3779035238316566,
 'Marriage 1': -0.2129589380669126,
 'Edu 1': -0.024214112522538807,
 'Edu 4': -0.0675666861694453,
 'Edu 5': -0.4351124817218204,
 'Edu 6': 1.242713126796456,
 'Edu 7': 0.16216826670661927}

In [16]:
n = X_validation.shape[0]
pred = lr.predict(X_validation) # Generate predictions for the validation set using the logistic regression model
TN, FP, FN, TP = confusion_matrix(y_validation, pred).ravel() # Extract true negatives, false positives, false negatives, and true positives from confusion matrix
print('Accuracy: ', accuracy_score(y_validation, pred)) # Calculate and print the accuracy of the model
print('Precision: ', precision_score(y_validation, pred)) # Calculate and print the precision of the model
print('Recall (True Positive): ', recall_score(y_validation, pred)) # Calculate and print the recall (true positive rate) of the model
print('True Negative Rate: ', TN/(TN+FP)) # Calculate and print the true negative rate (specificity) of the model
print('Sum of True Positive Rate and True Negative Rate', TP/(TP+FN)+TN/(TN+FP)) # Calculate and print the sum of true positive rate and true negative rate

Accuracy:  0.8466666666666667
Precision:  0.7948717948717948
Recall (True Positive):  0.8985507246376812
True Negative Rate:  0.8024691358024691
Sum of True Positive Rate and True Negative Rate 1.7010198604401503


In [17]:
# Evaluate model performance at different probability thresholds
Q = lr.predict_proba(X_validation)[:,1]
THRESHOLD = [0.4, 0.45, 0.5, 0.55, 0.6]
for i in THRESHOLD:
    print('\nTHRESOLD = ',i)
    pred = np.where(Q>i,1,0)
    TN, FP, FN, TP = confusion_matrix(y_validation, pred).ravel()
    print('Accuracy: ', accuracy_score(y_validation, pred))
    print('Precision: ', precision_score(y_validation, pred))
    print('Recall (True Positive): ', recall_score(y_validation, pred))
    print('True Negative Rate: ', TN/(TN+FP))
    print('Sum of True Positive Rate and True Negative Rate', TP/(TP+FN)+TN/(TN+FP))


THRESOLD =  0.4
Accuracy:  0.8333333333333334
Precision:  0.775
Recall (True Positive):  0.8985507246376812
True Negative Rate:  0.7777777777777778
Sum of True Positive Rate and True Negative Rate 1.6763285024154588

THRESOLD =  0.45
Accuracy:  0.84
Precision:  0.7848101265822784
Recall (True Positive):  0.8985507246376812
True Negative Rate:  0.7901234567901234
Sum of True Positive Rate and True Negative Rate 1.6886741814278046

THRESOLD =  0.5
Accuracy:  0.8466666666666667
Precision:  0.7948717948717948
Recall (True Positive):  0.8985507246376812
True Negative Rate:  0.8024691358024691
Sum of True Positive Rate and True Negative Rate 1.7010198604401503

THRESOLD =  0.55
Accuracy:  0.8466666666666667
Precision:  0.7948717948717948
Recall (True Positive):  0.8985507246376812
True Negative Rate:  0.8024691358024691
Sum of True Positive Rate and True Negative Rate 1.7010198604401503

THRESOLD =  0.6
Accuracy:  0.84
Precision:  0.8
Recall (True Positive):  0.8695652173913043
True Negativ

In [18]:
from sklearn.model_selection import GridSearchCV # Import GridSearchCV for hyperparameter tuning

In [19]:
# Try L2 regularization instead of L1 regularization
lr = LogisticRegression(penalty='l2' ,solver='liblinear') # L2 regularization parameter
params = {'C':[i/100 for i in range(1,101)]} # Define a range of values for the regularization strength parameter C

### The following is our grid search function, searching through all parameters in param. It looks for the parameters that maximizes accuracy precision, and recall. Since there are three things to improve on, we pick recall as the one that determines the best parameter. (This is an arbitrary choice)

In [21]:
clf = GridSearchCV(lr, params, cv=5, scoring='recall')

In [22]:
clf.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(solver='liblinear'),
             param_grid={'C': [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08,
                               0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16,
                               0.17, 0.18, 0.19, 0.2, 0.21, 0.22, 0.23, 0.24,
                               0.25, 0.26, 0.27, 0.28, 0.29, 0.3, ...]},
             scoring='recall')

In [23]:
clf.best_params_ # Retrieve the best hyperparameters found by GridSearchCV after tuning

{'C': 0.62}

In [24]:
lr = LogisticRegression(penalty='l2',solver='liblinear',C=0.8) # Initialize Logistic Regression model with L2 regularization and specified parameters
lr.fit(X_train, y_train) # Fit the Logistic Regression model to the training data
pred = lr.predict(X_validation) # Generate predictions for the validation set using the trained model
TN, FP, FN, TP = confusion_matrix(y_validation, pred).ravel()
print('Accuracy: ', accuracy_score(y_validation, pred))
print('Precision: ', precision_score(y_validation, pred))
print('Recall (True Positive): ', recall_score(y_validation, pred))
print('True Negative Rate: ', TN/(TN+FP))
print('Sum of True Positive Rate and True Negative Rate', TP/(TP+FN)+TN/(TN+FP))

Accuracy:  0.8533333333333334
Precision:  0.7974683544303798
Recall (True Positive):  0.9130434782608695
True Negative Rate:  0.8024691358024691
Sum of True Positive Rate and True Negative Rate 1.7155126140633388


In [25]:
# Save the trained Logistic Regression model to a file using pickle
import pickle
filename = 'finalized_model.sav'
pickle.dump(lr, open(filename, 'wb'))

### Test the Model Out-of-Sample

In [27]:
# delete all variables and functions in namespace
%reset -f
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

In [28]:
# Loading test and train data
test_data = pd.read_csv('test_data.csv',index_col=0)
train_data = pd.read_csv('train_data.csv',index_col=0)

# Convert "Occupation" into a dummy variable
test_data.loc[test_data['OCCUPATION']!=1, 'OCCUPATION'] = 0
train_data.loc[train_data['OCCUPATION']!=1, 'OCCUPATION'] = 0

total_data=pd.concat([train_data,test_data])
total_data=total_data.reset_index(drop=True)

In [29]:
# first, we list all the categorical variables to be one hot encoded
cat_vars = ['MARRIAGE', 'EDUCATION']

In [30]:
encoders = [OneHotEncoder(categories='auto') for _ in range(len(cat_vars))] # create an encoder for each cat_vars

# encode each of the cat_vars with their respective encoder
encoded_total = [encoders[i].fit_transform(total_data[[cat_var]]).todense() for i,cat_var in enumerate(cat_vars)]

In [31]:
# aggregating our data with the one hot encoded data
# drop the label column and also drop the cat vars
# this way we can join the encoded categorical variables with the continuous variables
X_total = pd.concat([total_data.iloc[:,:-1].drop(cat_vars, axis=1),
                     pd.DataFrame(np.concatenate(encoded_total, axis=1))], axis=1)
y_test = test_data.iloc[:,-1]
X_test = X_total.tail(len(y_test))
X_test = X_test.rename(columns={0: 'Marriage 1',1: 'Marriage 2' ,2: 'Marriage 3' ,3: 'Edu 1' ,4: 'Edu 2',5: 'Edu 3' ,
                                6: 'Edu 4',7: 'Edu 5',8: 'Edu 6' ,9: 'Edu 7'})
X_train = X_total[0:len(X_total) - len(y_test)]

In [32]:
# normalization on specific columns of a test dataset (X_test) based on the statistics calculated from a training dataset (X_train)
for i in [0,1,2,3,4,5,8]:
    X1 = X_train.iloc[:,1]
    mean = X1.mean()
    std = X1.std()
    X_test.iloc[:,i] = (X_test.iloc[:,i]-mean)/std

/var/folders/1h/mhjxr6bx0yg08w73dbfxx3vh0000gn/T/ipykernel_48038/3102543451.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '550    100.012609
551     -1.703906
552     -1.703906
553    167.823619
554     -1.703906
          ...    
673     32.201599
674     -1.703906
675     66.107104
676    133.918114
677    100.012609
Name: DEBT, Length: 128, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  X_test.iloc[:,i] = (X_test.iloc[:,i]-mean)/std
/var/folders/1h/mhjxr6bx0yg08w73dbfxx3vh0000gn/T/ipykernel_48038/3102543451.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '550   -1.025796
551   -0.008630
552   -0.008630
553    1.008535
554   -0.686741
         ...   
673   -1.364851
674   -1.025796
675    1.008535
676   -0.347686
677    1.008535
Name: YRS_IN_RESIDENT, Length: 128, dtype: float6

In [33]:
X_test.head()

,DEBT,YRS_IN_RESIDENT,AGE,YRS_OF_EMPLOYMENT,DTI,NUM_PREV_APP,OCCUPATION,PROVIDED_SIN,INCOME,CREDIT_PROFILE,Marriage 1,Marriage 2,Marriage 3,Edu 1,Edu 2,Edu 3,Edu 4,Edu 5,Edu 6,Edu 7
550,100.012609,-1.025796,15.926957,-1.025796,-1.195323,-1.703906,0,1,-1.364851,0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
551,-1.703906,-0.008630,3.720975,-0.771504,-1.478434,-1.364851,0,1,0.330425,0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
552,-1.703906,-0.008630,5.416250,-1.619142,-1.619142,-1.703906,0,1,1.008535,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
553,167.823619,1.008535,13.892627,1.856172,-0.008630,0.669480,1,0,2.025700,1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
554,-1.703906,-0.686741,8.467746,-1.025796,-1.690344,-1.703906,0,1,3.042865,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [34]:
# Load the previously saved Logistic Regression model from a file
import pickle
lr_load = pickle.load(open('finalized_model.sav', 'rb'))

In [35]:
# process helps in understanding how well the model performs on data.
pred = lr_load.predict(X_test)
TN, FP, FN, TP = confusion_matrix(y_test, pred).ravel()
print('Accuracy: ', accuracy_score(y_test, pred))
print('Precision: ', precision_score(y_test, pred))
print('Recall (True Positive): ', recall_score(y_test, pred))
print('True Negative Rate: ', TN/(TN+FP))
print('Sum of True Positive Rate and True Negative Rate', TP/(TP+FN)+TN/(TN+FP))

Accuracy:  0.5625
Precision:  0.5045871559633027
Recall (True Positive):  0.9649122807017544
True Negative Rate:  0.23943661971830985
Sum of True Positive Rate and True Negative Rate 1.2043489004200643


### Interpretation of the results

Based on the results calculated above, we can see that around 56.25% of the model were correctly predicted. The precision of around 50.46% indicated the proportion of positive predictions that were actually correct. The recall of 96.49% shows that the model successfully got correct 96.49% of the actual positive cases. The true negative rate 23.94% is the rate at which the model was able to identify true negative cases. The low rate shows that the model may be biased in identifying the positive cases rather than the negative cases. However, the sum of the recall and true negative rate of 1.2043 is promising as a sum over 1 indicates a balanced model.